In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.process_tree.obj import ProcessTree
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.objects.petri_net.exporter import exporter as pnml_exporter

# INDUCTIVE MINER 

xes_file = r"INSERT XES PATH"
output_png = r"INSERT PNG OUTPUT PATH"

log = xes_importer.apply(xes_file)

res = inductive_miner.apply(log, variant=inductive_miner.Variants.IM)

if isinstance(res, ProcessTree):
    net, initial_marking, final_marking = pt_converter.apply(
        res, variant=pt_converter.Variants.TO_PETRI_NET
    )
else:
    net, initial_marking, final_marking = res

parameters = {
    pn_visualizer.Variants.FREQUENCY.value.Parameters.FORMAT: "png"
}

gviz = pn_visualizer.apply(net, initial_marking, final_marking, parameters=parameters)
pn_visualizer.save(gviz, output_png)



parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
Model discovered
Petri net saved as PNG


In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.visualization.petri_net import visualizer as pn_visualizer

# HEURISTIC MINER 

xes_file = r"INSERT XES PATH"
output_png = r"INSERT PNG OUTPUT PATH"

log = xes_importer.apply(xes_file)

parameters = {
    heuristics_miner.Variants.CLASSIC.value.Parameters.DEPENDENCY_THRESH: 0.95,
    heuristics_miner.Variants.CLASSIC.value.Parameters.AND_MEASURE_THRESH: 0.9,
    heuristics_miner.Variants.CLASSIC.value.Parameters.LOOP_LENGTH_TWO_THRESH: 0.95,
    heuristics_miner.Variants.CLASSIC.value.Parameters.MIN_ACT_COUNT: 3,
    heuristics_miner.Variants.CLASSIC.value.Parameters.MIN_DFG_OCCURRENCES: 4,
}

net, initial_marking, final_marking = heuristics_miner.apply(log, parameters=parameters)

gviz = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(gviz, output_png)


parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
Heuristic miner model discovered
Petri net saved to C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_heuristic.png


In [ ]:
import pm4py
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.objects.conversion.process_tree import converter as pt_converter
from pm4py.objects.process_tree.obj import ProcessTree
from pm4py.visualization.petri_net import visualizer as pn_visualizer

from pm4py.objects.petri_net.exporter import exporter as pnml_exporter

# INDUCTIVE INFREQUENT MINER

xes_file = r"INSERT XES PATH"
output_png = r"INSERT PNG OUTPUT PATH"

log = xes_importer.apply(xes_file)

parameters = {"noiseThreshold": 0.9} 
res = inductive_miner.apply(log, variant=inductive_miner.Variants.IMf, parameters=parameters)

if isinstance(res, ProcessTree):
    net, initial_marking, final_marking = pt_converter.apply(
        res, variant=pt_converter.Variants.TO_PETRI_NET
    )
else:
    net, initial_marking, final_marking = res

gviz = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(gviz, output_png)


parsing log, completed traces ::   0%|          | 0/900 [00:00<?, ?it/s]

Log Imported
IMf miner model discovered
Model discovered
Petri net saved as PNG to C:\Users\lomo0\Downloads\900SmallerLogs\smaller_pn_imf3_09.png


In [ ]:
import xml.etree.ElementTree as ET

# XES CLEANING

INPUT_XES = "INSERT XES PATH"
OUTPUT_XES = "INSERT XES OUTPUT PATH"

tree = ET.parse(INPUT_XES)
root = tree.getroot()

ns = {"xes": "http://www.xes-standard.org/"}

def is_login(event_elem):
    concept_name_elem = event_elem.find("xes:string[@key='concept:name']", ns)
    if concept_name_elem is not None:
        return concept_name_elem.attrib["value"].startswith("4624")
    return False

traces_to_remove = []

# Remove all events before the first login event in each trace.
for trace in root.findall("xes:trace", ns):
    events = trace.findall("xes:event", ns)
    login_found = False
    new_events = []

    for event in events:
        if not login_found and is_login(event):
            login_found = True 
        if login_found:
            new_events.append(event)

    for e in events:
        trace.remove(e)

    for e in new_events:
        trace.append(e)

    if not new_events:
        traces_to_remove.append(trace)

for trace in traces_to_remove:
    root.remove(trace)

print(f"Removed {len(traces_to_remove)} traces")
print(f"Remaining traces: {len(root.findall('xes:trace', ns))}")

def indent(elem, level=0):
    i = "\n" + level*"  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        for child in elem:
            indent(child, level+1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if not elem.tail or not elem.tail.strip():
            elem.tail = i

indent(root)
ET.register_namespace('', "http://www.xes-standard.org/")
tree.write(OUTPUT_XES, encoding="UTF-8", xml_declaration=True)

Removed 536 traces with no login events
Remaining traces: 60829
✅ Cleaned XES written to C:/Users/lomo0/Downloads/wls_processtype_cleaned.xes
